In [1]:
import os
import glob
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import numpy as np
from tqdm import tqdm
import json

In [2]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

Using device: cuda


In [3]:
with open('config.json', 'r') as config_file:
    config = json.load(config_file)

In [4]:
TRAIN_DIR = config['data']['train_dir']
VALID_DIR = config['data']['valid_dir']
RESIZED_CLEAN_TRAIN_DIR = config['data']['resized_clean_train_dir']
RESIZED_CLEAN_VALID_DIR = config['data']['resized_clean_valid_dir']
RESIZED_NOISY_TRAIN_DIR = config['data']['resized_noisy_train_dir']
RESIZED_NOISY_VALID_DIR = config['data']['resized_noisy_valid_dir']


In [5]:
RESIZE_SIZE = tuple(config['image_processing']['resize_size'])
NOISE_STD = config['image_processing']['noise_std']

def resize_and_save_images(input_dir, output_dir):
    files = glob.glob(os.path.join(input_dir, '*.png'))
    for img_path in tqdm(files, desc=f'Resizing images in {input_dir}'): 
        image = Image.open(img_path).convert('RGB')
        image = image.resize(RESIZE_SIZE)
        image.save(os.path.join(output_dir, os.path.basename(img_path)))

def resize_add_noise_and_save_images(input_dir, output_dir, noise_std=NOISE_STD):
    files = glob.glob(os.path.join(input_dir, '*.png'))
    for img_path in tqdm(files, desc=f'Resizing and adding noise to images in {input_dir}'): 
        image = Image.open(img_path).convert('RGB')
        image = image.resize(RESIZE_SIZE)
        
        noisy_image = add_gaussian_noise(image, std=noise_std)
        noisy_image.save(os.path.join(output_dir, os.path.basename(img_path)))

def add_gaussian_noise(image, mean=0.0, std=25.0):
    image_np = np.array(image, dtype=np.float32)
    noise = np.random.normal(mean, std, image_np.shape).astype(np.float32)
    noisy_image = image_np + noise
    return Image.fromarray(np.clip(noisy_image, 0, 255).astype(np.uint8))

In [6]:
print(os.path.isdir('DIV2K'))

True


In [7]:
if not os.path.isdir('./DIV2K'):
    os.makedirs(RESIZED_CLEAN_TRAIN_DIR, exist_ok=True)
    os.makedirs(RESIZED_CLEAN_VALID_DIR, exist_ok=True)
    os.makedirs(RESIZED_NOISY_TRAIN_DIR, exist_ok=True)
    os.makedirs(RESIZED_NOISY_VALID_DIR, exist_ok=True)
    
    resize_and_save_images(TRAIN_DIR, RESIZED_CLEAN_TRAIN_DIR)
    resize_and_save_images(VALID_DIR, RESIZED_CLEAN_VALID_DIR)
    resize_add_noise_and_save_images(TRAIN_DIR, RESIZED_NOISY_TRAIN_DIR)
    resize_add_noise_and_save_images(VALID_DIR, RESIZED_NOISY_VALID_DIR)

In [8]:
class DIV2KDataset(Dataset):
    def __init__(self, clean_dir, noisy_dir=None, transform=None):
        self.clean_files = glob.glob(os.path.join(clean_dir, '*.png'))
        self.noisy_files = glob.glob(os.path.join(noisy_dir, '*.png')) if noisy_dir else None
        self.transform = transform

    def __len__(self):
        return len(self.clean_files)

    def __getitem__(self, idx):
        clean_img_path = self.clean_files[idx]
        clean_image = Image.open(clean_img_path).convert('RGB')
        
        if self.noisy_files:
            noisy_img_path = self.noisy_files[idx]
            noisy_image = Image.open(noisy_img_path).convert('RGB')
        else:
            noisy_image = clean_image
        
        if self.transform:
            clean_image = self.transform(clean_image)
            noisy_image = self.transform(noisy_image)
        
        return noisy_image, clean_image

In [9]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

train_dataset = DIV2KDataset(RESIZED_CLEAN_TRAIN_DIR, RESIZED_NOISY_TRAIN_DIR, transform=transform)
valid_dataset = DIV2KDataset(RESIZED_CLEAN_VALID_DIR, RESIZED_NOISY_VALID_DIR, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True, num_workers=0)
valid_loader = DataLoader(valid_dataset, batch_size=4, shuffle=False, num_workers=0)


In [ ]:
'''


class DenoisingAutoencoder(nn.Module):
    def __init__(self):
        super(DenoisingAutoencoder, self).__init__()
        
   
        self.encoder = nn.Sequential(
            nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.Conv2d(64, 32, kernel_size=3, stride=2, padding=1),
            nn.ReLU(),
            nn.Conv2d(32, 16, kernel_size=3, stride=2, padding=1),
            nn.ReLU()
        )
        
  
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(16, 32, kernel_size=3, stride=2, padding=1, output_padding=1),
            nn.ReLU(),
            nn.ConvTranspose2d(32, 64, kernel_size=3, stride=2, padding=1, output_padding=1),
            nn.ReLU(),
            nn.Conv2d(64, 3, kernel_size=3, stride=1, padding=1),
            nn.Tanh()
        )
        
    def forward(self, x):
        x = self.encoder(x)
        x = self.decoder(x)
        return x

'''

class DenoisingAutoencoder(nn.Module):
    def __init__(self):
        super(DenoisingAutoencoder, self).__init__()

        self.encoder = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Conv2d(32, 32, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )

        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(32, 32, kernel_size=2, stride=2),
            nn.ReLU(),
            nn.ConvTranspose2d(32, 32, kernel_size=2, stride=2),
            nn.ReLU(),
            nn.Conv2d(32, 3, kernel_size=3, stride=1, padding=1),
            nn.Sigmoid()  #sigmoid for [0, 1]
        )
        
    def forward(self, x):
        x = self.encoder(x)
        x = self.decoder(x)
        return x



In [11]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = DenoisingAutoencoder().to(device)
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.003)

In [12]:
def train_model(model, train_loader, valid_loader, criterion, optimizer, num_epochs):
    for epoch in range(num_epochs):
        model.train()
        train_loss = 0.0

        for noisy_imgs, clean_imgs in tqdm(train_loader, desc=f'Training Epoch {epoch+1}/{num_epochs}', unit='batch'):
            noisy_imgs, clean_imgs = noisy_imgs.to(device), clean_imgs.to(device)
            
            outputs = model(noisy_imgs)
            loss = criterion(outputs, clean_imgs)
            
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
            train_loss += loss.item() * noisy_imgs.size(0)
        
        train_loss /= len(train_loader.dataset)

        model.eval()
        valid_loss = 0.0
        with torch.no_grad():
            for noisy_imgs, clean_imgs in tqdm(valid_loader, desc=f'Validation Epoch {epoch+1}/{num_epochs}', unit='batch'):
                noisy_imgs, clean_imgs = noisy_imgs.to(device), clean_imgs.to(device)
                outputs = model(noisy_imgs)
                loss = criterion(outputs, clean_imgs)
                valid_loss += loss.item() * noisy_imgs.size(0)
        valid_loss /= len(valid_loader.dataset)
        
        print(f'Epoch [{epoch+1}/{num_epochs}], Train Loss: {train_loss:.4f}, Valid Loss: {valid_loss:.4f}')

train_model(model, train_loader, valid_loader, criterion, optimizer, num_epochs=10)


Validation Epoch 1/10: 100%|██████████| 25/25 [00:11<00:00,  2.22batch/s]


Epoch [1/10], Train Loss: 0.2764, Valid Loss: 0.2616


Validation Epoch 2/10: 100%|██████████| 25/25 [00:08<00:00,  2.90batch/s]


Epoch [2/10], Train Loss: 0.2398, Valid Loss: 0.2387


Validation Epoch 3/10: 100%|██████████| 25/25 [00:09<00:00,  2.70batch/s]


...

Epoch [8/10], Train Loss: 0.2266, Valid Loss: 0.2345


Validation Epoch 9/10: 100%|██████████| 25/25 [00:10<00:00,  2.50batch/s]


Epoch [9/10], Train Loss: 0.2260, Valid Loss: 0.2341


Validation Epoch 10/10: 100%|██████████| 25/25 [00:09<00:00,  2.67batch/s]

Epoch [10/10], Train Loss: 0.2257, Valid Loss: 0.2341


In [13]:
torch.save(model, "model.pth")